# Model Training

End-to-end training pipeline for the cross-sell propensity model:
Load → Feature Engineering → Tune → Train → Evaluate → Save → Log to MLflow

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

from src.data import load_config, load_raw, clean, split
from src.features import build_feature_matrix
from src.model import tune_hyperparameters, train_model, tune_threshold, evaluate, save_model

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

cfg       = load_config("../config.yaml")
TARGET    = cfg["data"]["target_col"]
DROP_COLS = ["id", TARGET]

## 2. Load & Engineer Features

In [ ]:
df = clean(load_raw("../data/raw"))
train_df, test_df = split(
    df,
    test_size=cfg["data"]["test_size"],
    random_state=cfg["split"]["random_state"],
)

# Stateless transforms — freq/target encoding handled inside the pipeline
train_eng = build_feature_matrix(train_df)
test_eng  = build_feature_matrix(test_df)

X_train = train_eng.drop(columns=DROP_COLS)
y_train = train_eng[TARGET]
X_test  = test_eng.drop(columns=DROP_COLS)
y_test  = test_eng[TARGET]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 3. Hyperparameter Tuning

In [ ]:
best_params = tune_hyperparameters(
    X_train, y_train,
    n_trials=cfg["tuning"]["n_trials"],
    cv_folds=cfg["tuning"]["cv_folds"],
    random_state=cfg["split"]["random_state"],
)

print("\nBest hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

## 4. Train Final Model

In [ ]:
pipeline = train_model(
    X_train, y_train,
    best_params,
    random_state=cfg["split"]["random_state"],
)
print("Pipeline trained:", pipeline)

## 5. Evaluate

In [ ]:
probs     = pipeline.predict_proba(X_test)[:, 1]
threshold = tune_threshold(y_test, probs, beta=2.0)
result    = evaluate("XGBoost (tuned)", y_test, probs, threshold=threshold)

pd.DataFrame([result]).set_index("model")

## 6. PR & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

PrecisionRecallDisplay.from_predictions(y_test, probs, name="XGBoost (tuned)", ax=axes[0])
axes[0].axhline(y_test.mean(), color="grey", linestyle="--", label=f"Random ({y_test.mean():.2f})")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend()

RocCurveDisplay.from_predictions(y_test, probs, name="XGBoost (tuned)", ax=axes[1])
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()

In [ ]:
all_results = results + [eng_result, tuned_result]
final_df    = pd.DataFrame(all_results).set_index("model")

print("=" * 60)
print("FULL RESULTS COMPARISON")
print("=" * 60)
print(final_df.to_string())

baseline_pr = results[1]["pr_auc"]
print(f"\nPR-AUC lift — engineered vs baseline : {(eng_result['pr_auc']  - baseline_pr) / baseline_pr * 100:+.1f}%")
print(f"PR-AUC lift — tuned vs baseline      : {(tuned_result['pr_auc'] - baseline_pr) / baseline_pr * 100:+.1f}%")

## 7. Save Model

In [ ]:
save_model(pipeline, threshold, path=cfg["paths"]["model_file"])

## 8. Log to MLflow

In [ ]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("cross-sell-propensity")

with mlflow.start_run(run_name="XGBoost (tuned)"):
    mlflow.set_tags({
        "stage":   "tuned",
        "features": "engineered + pipeline (freq + target encoding)",
        "dataset":  "health-insurance-cross-sell",
    })

    mlflow.log_params({
        "model_type": "XGBoost",
        **{k: round(v, 4) if isinstance(v, float) else v for k, v in best_params.items()},
    })

    mlflow.log_metrics({
        "pr_auc":    result["pr_auc"],
        "roc_auc":   result["roc_auc"],
        "precision": result["precision"],
        "recall":    result["recall"],
        "threshold": result["threshold"],
    })

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

    print(f"✓ Logged run | PR-AUC: {result['pr_auc']} | ROC-AUC: {result['roc_auc']}")
    print(f"Runs saved to: {mlflow.get_tracking_uri()}")